In [1]:
import os
from dotenv import load_dotenv

In [2]:
# Import LangChain components
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain.vectorstores import Chroma
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
from langchain.prompts import PromptTemplate

In [3]:
# Load the API key from the .env file
load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")

# Check if the API key is available
if not api_key:
    raise ValueError("Google API Key not found. Please set it in the .env file.")
else:
    print("Libraries imported and API key loaded successfully.")

Libraries imported and API key loaded successfully.


In [ ]:

PDF_DOCS_PATH = "./documents"
CHROMA_PERSIST_PATH = "./db/chroma_store"

# Prompt for llm
custom_prompt_template = """
You are a helpful and honest medical information assistant.
Your task is to provide answers based *only* on the provided context.
If the information is not in the context, you must respond with "I am sorry, but I do not have information on that specific topic in my knowledge base."
Do not make up information or use any external knowledge.
Your answers should be clear and concise.

Context: {context}
Chat History: {chat_history}

Question: {question}
Helpful Answer:
"""

def create_custom_prompt():
    """Creates a custom prompt template for the RAG chain."""
    return PromptTemplate(
        template=custom_prompt_template,
        input_variables=["context", "chat_history", "question"]
    )

print("Constants and prompt template function defined.")

Constants and prompt template function defined.


In [ ]:

def get_vectorstore_with_throttling():
    """
    Processes PDF documents with a delay between embedding calls to avoid rate limiting.
    Creates/loads a persistent Chroma vector store.
    """
    # Initialize the embedding model
    embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001", google_api_key=api_key)

    if os.path.exists(CHROMA_PERSIST_PATH):
        # If the store already exists, just load it
        print("Loading existing vector store...")
        vector_store = Chroma(persist_directory=CHROMA_PERSIST_PATH, embedding_function=embeddings)
    else:
        print("Creating new vector store with throttling...")
        
        # 1. Load and chunk the documents as before
        pdf_files = [f for f in os.listdir(PDF_DOCS_PATH) if f.endswith(".pdf")]
        if not pdf_files:
            raise FileNotFoundError(f"No PDF files found in '{PDF_DOCS_PATH}'.")

        documents = []
        for pdf_file in pdf_files:
            loader = PyPDFLoader(os.path.join(PDF_DOCS_PATH, pdf_file))
            documents.extend(loader.load())
        
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
        chunks = text_splitter.split_documents(documents)
        total_chunks = len(chunks)
        print(f"Total chunks to be processed: {total_chunks}")

        # 2. Initialize an empty Chroma vector store
        vector_store = Chroma(
            embedding_function=embeddings,
            persist_directory=CHROMA_PERSIST_PATH
        )

        # 3. Add chunks to the store one by one with a delay
        for i, chunk in enumerate(chunks):
            # The add_documents method takes a list of documents
            vector_store.add_documents([chunk])
            
            # Print progress and wait for 1 second
            print(f"  > Embedded chunk {i + 1} of {total_chunks}")
            time.sleep(1) 
        
        print("\nVector store created successfully!")

    return vector_store

In [7]:
#Conversational RAG Chain Function 

def get_conversational_rag_chain(vector_store):
    """
    Creates the main conversational retrieval chain with memory.
    """
    # Initialize the Gemini LLM
    llm = ChatGoogleGenerativeAI(model="gemini-pro", google_api_key=api_key, temperature=0.3)

    # Set up memory to store chat history
    memory = ConversationBufferMemory(
        memory_key="chat_history",
        return_messages=True
    )

    # Create the retriever from the vector store
    retriever = vector_store.as_retriever(search_kwargs={"k": 3}) # Retrieve top 3 relevant chunks

    # Create the custom prompt
    prompt = create_custom_prompt()

    # Create the conversational chain
    chain = ConversationalRetrievalChain.from_llm(
        llm=llm,
        retriever=retriever,
        memory=memory,
        combine_docs_chain_kwargs={"prompt": prompt},
        verbose=False # Set to True to see detailed chain activity
    )
    
    return chain

print("Conversational RAG chain function defined.")

Conversational RAG chain function defined.


In [ ]:
# Initialization

vector_store = get_vectorstore_with_throttling()
qa_chain = get_conversational_rag_chain(vector_store)

print("✅ Chatbot is ready! You can now ask questions in the next cell.")

Creating new vector store with throttling...
Total chunks to be processed: 371


C:\Users\akmal\AppData\Local\Temp\ipykernel_27088\3065986731.py:33: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vector_store = Chroma(


GoogleGenerativeAIError: Error embedding content: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
  quota_metric: "generativelanguage.googleapis.com/embed_content_free_tier_requests"
  quota_id: "EmbedContentRequestsPerDayPerUserPerProjectPerModel-FreeTier"
}
violations {
  quota_metric: "generativelanguage.googleapis.com/embed_content_free_tier_requests"
  quota_id: "EmbedContentRequestsPerMinutePerUserPerProjectPerModel-FreeTier"
}
violations {
  quota_metric: "generativelanguage.googleapis.com/embed_content_free_tier_requests"
  quota_id: "EmbedContentRequestsPerMinutePerProjectPerModel-FreeTier"
}
violations {
  quota_metric: "generativelanguage.googleapis.com/embed_content_free_tier_requests"
  quota_id: "EmbedContentRequestsPerDayPerProjectPerModel-FreeTier"
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai

In [ ]:
# Interactive Chat 

try:
    chat_history
except NameError:
    chat_history = []

# --- Ask qns ---
question = "What are the early symptoms of Type 2 Diabetes?"
# --------------------------

if question.lower() == 'exit':
    print("Chat ended.")
else:
    # Get the result from the chain
    result = qa_chain({"question": question, "chat_history": chat_history})
    
    # Append the current question and the answer to the history
    chat_history.append((question, result["answer"]))
    
    # Print the answer
    print("AI:", result["answer"])

C:\Users\akmal\AppData\Local\Temp\ipykernel_25232\563631256.py:18: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = qa_chain({"question": question, "chat_history": chat_history})


GoogleGenerativeAIError: Error embedding content: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
  quota_metric: "generativelanguage.googleapis.com/embed_content_free_tier_requests"
  quota_id: "EmbedContentRequestsPerDayPerProjectPerModel-FreeTier"
}
violations {
  quota_metric: "generativelanguage.googleapis.com/embed_content_free_tier_requests"
  quota_id: "EmbedContentRequestsPerMinutePerProjectPerModel-FreeTier"
}
violations {
  quota_metric: "generativelanguage.googleapis.com/embed_content_free_tier_requests"
  quota_id: "EmbedContentRequestsPerMinutePerUserPerProjectPerModel-FreeTier"
}
violations {
  quota_metric: "generativelanguage.googleapis.com/embed_content_free_tier_requests"
  quota_id: "EmbedContentRequestsPerDayPerUserPerProjectPerModel-FreeTier"
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai

In [12]:
# ### Cell: Document Analyzer ###

import os
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Make sure this path is correct for your project structure
PDF_DOCS_PATH = "./documents"

print(f"Analyzing documents in: {PDF_DOCS_PATH}\n")

# --- 1. Load the Documents ---
# Check if the path exists
if not os.path.exists(PDF_DOCS_PATH):
    print(f"🔴 ERROR: The directory '{PDF_DOCS_PATH}' was not found.")
else:
    pdf_files = [f for f in os.listdir(PDF_DOCS_PATH) if f.endswith(".pdf")]
    if not pdf_files:
        print(f"🟡 WARNING: No PDF files found in '{PDF_DOCS_PATH}'.")
    else:
        documents = []
        for pdf_file in pdf_files:
            loader = PyPDFLoader(os.path.join(PDF_DOCS_PATH, pdf_file))
            documents.extend(loader.load())

        # --- 2. Calculate Total Characters and Words ---
        total_text = "".join(doc.page_content for doc in documents)
        total_characters = len(total_text)
        total_words = len(total_text.split())

        print(f"--- Document Stats ---")
        print(f"Total number of loaded pages: {len(documents)}")
        print(f"Total number of characters:   {total_characters:,}")
        print(f"Estimated total number of words:  {total_words:,}\n")

        # --- 3. Calculate the Number of Chunks ---
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
        chunks = text_splitter.split_documents(documents)
        total_chunks = len(chunks)

        print(f"--- Chunking Analysis ---")
        print(f"With a chunk size of 1000, your documents will be split into:")
        print(f"==> Total Chunks: {total_chunks}")
        print("\nThis means your script will try to make roughly", total_chunks, "API calls to create the embeddings.")

Analyzing documents in: ./documents



Advanced encoding /SymbolSetEncoding not implemented yet
Advanced encoding /SymbolSetEncoding not implemented yet
Advanced encoding /SymbolSetEncoding not implemented yet
Advanced encoding /SymbolSetEncoding not implemented yet
Advanced encoding /SymbolSetEncoding not implemented yet
Advanced encoding /SymbolSetEncoding not implemented yet
Advanced encoding /SymbolSetEncoding not implemented yet
Advanced encoding /SymbolSetEncoding not implemented yet
Advanced encoding /SymbolSetEncoding not implemented yet
Advanced encoding /SymbolSetEncoding not implemented yet
Advanced encoding /SymbolSetEncoding not implemented yet
Advanced encoding /SymbolSetEncoding not implemented yet
Advanced encoding /SymbolSetEncoding not implemented yet
Advanced encoding /SymbolSetEncoding not implemented yet
Advanced encoding /SymbolSetEncoding not implemented yet
Advanced encoding /SymbolSetEncoding not implemented yet
Advanced encoding /SymbolSetEncoding not implemented yet
Advanced encoding /SymbolSetEnc

--- Document Stats ---
Total number of loaded pages: 540
Total number of characters:   1,094,051
Estimated total number of words:  163,800

--- Chunking Analysis ---
With a chunk size of 1000, your documents will be split into:
==> Total Chunks: 1458

This means your script will try to make roughly 1458 API calls to create the embeddings.
